In [0]:
# Load transaction data
df = spark.read.csv("/Volumes/workspace/default/loan/transaction_data.csv", header=True, inferSchema=True)

# Assume flagged nodes are known (e.g., flagged UserIds)
flagged_nodes = [267099, 350805, 372750]  # Example flagged UserIds

# Calculate graph connectivity to flagged nodes
from pyspark.sql.functions import col, countDistinct, sum, when, expr

# Count how many flagged nodes each receiver (UserId) is connected to
connectivity_df = df.filter(col("UserId").isin(flagged_nodes)) \
    .groupBy("ItemCode") \
    .agg(countDistinct("UserId").alias("flagged_connections"))

# Calculate velocity of incoming funds (total value received per receiver)
velocity_df = df.groupBy("ItemCode") \
    .agg(
        sum(col("NumberOfItemsPurchased") * col("CostPerItem")).alias("total_incoming_funds"),
        countDistinct("TransactionId").alias("transaction_count")
    )

# Identify first-time interactions (receivers with only one transaction)
first_time_df = df.groupBy("ItemCode") \
    .agg(countDistinct("TransactionId").alias("transaction_count")) \
    .filter(col("transaction_count") == 1) \
    .select("ItemCode")

# Combine risk factors
risk_df = connectivity_df.join(velocity_df, "ItemCode", "left") \
    .withColumn("first_time_interaction", when(col("ItemCode").isin([row.ItemCode for row in first_time_df.collect()]), 1).otherwise(0)) \
    .withColumn("risk_score", 
        expr("""
            flagged_connections * 2 +
            (total_incoming_funds / 1000) +
            first_time_interaction * 3
        """)
    ) \
    .withColumn("reason",
        when(col("flagged_connections") > 0, "Linked to high-risk nodes")
        .when(col("first_time_interaction") == 1, "First-time interaction")
        .otherwise("Normal")
    )

# Dashboard: Show receiver risk scores and reasons
display(risk_df.select("ItemCode", "risk_score", "reason", "flagged_connections", "total_incoming_funds", "first_time_interaction"))